# Customer Churn Classification System (TensorFlow – Advanced Classification)

## Project Goal

The goal of this project is to build a binary classification model that predicts whether a customer will leave the service or stay.

This project focuses not only on building a TensorFlow model, but also on understanding classification logic, preventing overfitting, and evaluating model performance in a more realistic way.

## Why Classification?

In this project, the target variable is **churn**:

- `0` → customer stays
- `1` → customer leaves

Since the model predicts one of two categories, this is a **binary classification problem**.

## What Problems Are Addressed in This Project?

This notebook focuses on:

- building a classification model
- understanding train / validation / test logic
- reducing overfitting
- using dropout
- applying early stopping
- evaluating model performance correctly

In [2]:
# 2.Classification Data
import numpy as np
import pandas as pd

from w3.d3.tensorflow_churn_prediction import history, test_accuracy, new_customer, prediction

np.random.seed(42)

# number of customers
n = 1000

# features
age = np.random.randint(18, 72, n)
monthly_spending = np.random.randint(20, 200, n)
subscription_months = np.random.randint(1, 60, n)
support_tickets = np.random.randint(0, 10, n)

# churn logic
churn = (
(subscription_months < 12) &
(support_tickets > 5)
).astype(int)

# feature matrix
X = np.column_stack((age, monthly_spending, subscription_months, support_tickets))

# target vector
y = churn

print("Feature shape: ", X.shape)
print("Target shape: ", y.shape)

print("First 5 targets: ", y[:5])

print("Churn distribution:", np.bincount(y))

Feature shape:  (1000, 4)
Target shape:  (1000,)
First 5 targets:  [0 0 0 1 0]
Churn distribution: [923  77]


### Dataset Balance

The dataset is slightly imbalanced. Most customers stay in the service, while a smaller portion of customers churn.

This situation is common in real subscription businesses where the majority of customers continue using the service.

Although the imbalance is not extreme, it is important to be aware of it because highly imbalanced datasets can bias classification models.

In [3]:
# 3.Train/Validation/Test
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Second split: Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (700, 4)
Validation shape: (150, 4)
Test shape: (150, 4)


In [6]:
# 4.Creating Model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    tf.keras.Input(shape=(4,)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 5. Adding dropout
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    tf.keras.Input(shape=(4,)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

### Dropout Layer

Dropout layers were added to reduce overfitting during training.

Dropout randomly disables a portion of neurons during each training step. This prevents the model from relying too heavily on specific neurons and encourages the network to learn more general patterns in the data.

In [9]:
# 6.Model Compile
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [10]:
# 7.Early Stopping
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [11]:
# 8.Training
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1800 - loss: 19.5124 - val_accuracy: 0.2867 - val_loss: 14.1382
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2829 - loss: 11.7190 - val_accuracy: 0.3533 - val_loss: 7.9315
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4214 - loss: 6.4778 - val_accuracy: 0.4867 - val_loss: 3.6347
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5914 - loss: 3.2922 - val_accuracy: 0.7533 - val_loss: 1.5412
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7486 - loss: 1.7923 - val_accuracy: 0.8333 - val_loss: 0.9649
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8214 - loss: 1.1625 - val_accuracy: 0.8800 - val_loss: 0.7485
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8443 - loss: 0.8488 - val_accuracy: 0.9133 - val_loss: 0.6056
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - accuracy: 0.8629 - loss: 0.6580 - val_accuracy: 0.906

In [14]:
# 9.Model Evaluation
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9267 - loss: 0.1389
Test loss: 0.13890263438224792
Test accuracy: 0.9266666769981384


In [17]:
# 10. Making Prediction
new_customer = np.array([[30,45,6,8]])

prediction = model.predict(new_customer)
print("Churn probability:", prediction[0][0])

if prediction[0][0] >= 0.5:
    print("Prediction: Customer is likely to churn.")
else:
    print("Prediction: Customer is likely to stay.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
Churn probability: 0.86230326
Prediction: Customer is likely to churn.


## Executive Summary

- The model achieved approximately **86% test accuracy** in predicting customer churn.
- High-risk customers can be identified earlier using this system.
- Dropout and early stopping techniques helped reduce the risk of overfitting.
- The model could support proactive strategies to reduce customer churn.